# DEV — Train & Track with MLflow

**Duration:** 5 min

## Project Structure

In [ ]:
churn-mlops/
├── data/
│   └── telco_churn.csv
├── src/
│   ├── train.py          # training script
│   ├── predict.py        # inference logic
│   └── preprocess.py     # feature engineering
├── api/
│   └── app.py            # FastAPI app
├── tests/
│   └── test_api.py       # integration tests
├── Dockerfile
├── .github/workflows/
│   └── ci.yml
└── requirements.txt

> **Try it in Google Colab:** [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shastrula/ailearningclub-courses/blob/main/mlops-real-world/mlops-dev-train.ipynb)


## Preprocess & Train

In [ ]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, f1_score
import joblib

mlflow.set_experiment('churn-prediction')

df = pd.read_csv('data/telco_churn.csv')
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)
df['Churn'] = (df['Churn'] == 'Yes').astype(int)

# Encode categoricals
cat_cols = df.select_dtypes('object').columns.drop('customerID')
le = LabelEncoder()
for col in cat_cols:
    df[col] = le.fit_transform(df[col])

X = df.drop(columns=['customerID','Churn'])
y = df['Churn']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,
                                                     stratify=y, random_state=42)

params = {'n_estimators': 200, 'max_depth': 4, 'learning_rate': 0.05, 'subsample': 0.8}

with mlflow.start_run():
    mlflow.log_params(params)
    model = GradientBoostingClassifier(**params, random_state=42)
    model.fit(X_train, y_train)

    probs = model.predict_proba(X_test)[:, 1]
    preds = (probs >= 0.5).astype(int)
    auc = roc_auc_score(y_test, probs)
    f1  = f1_score(y_test, preds)

    mlflow.log_metrics({'auc': auc, 'f1': f1})
    mlflow.sklearn.log_model(model, 'model')
    print(f'AUC: {auc:.4f}  F1: {f1:.4f}')

joblib.dump({'model': model, 'features': list(X.columns)}, 'model.pkl')

```
AUC: 0.8541  F1: 0.6203
MLflow run logged to ./mlruns
```

> **💡 Tip:** Run `mlflow ui` to open the experiment dashboard at http://localhost:5000 — compare runs, view metrics, download artifacts.